In [1]:
from src.data_preprocessing import *

Подготовим ряды с `2020-01` по `2021-07`

Так как мы работаем с временными рядами, то лучше иметь информацию по всем месяцам. **Поэтому в дате начала и конца мы не указываем дни**

Все скрипты были подготовлены заранее и находятся в модуле `data_preprocessing.py`

Также имеется имплементация на PySpark в модуле `data_preprocessing_pyspark.py`

In [2]:
# Основные параметры 
DATE_START = '2020-01'
DATE_END = '2021-07'
PROJECT_SOURCE = '' # укажи путь к папке с проектом 

In [3]:
# Обработка геоданных (полигоны) 
polygon_data = preprocess_polygon_data('taxi_zones_polygons.geojson', file_path=PROJECT_SOURCE)
UNIQUE_DISTRICTS = polygon_data['location_id'].unique().tolist()

In [4]:
# ETL
f_path = PROJECT_SOURCE+'/taxi_data'
trips_df, mean_trips_df = get_series_data(file_path=f_path,
                                          start_date=DATE_START, 
                                          end_date=DATE_END,
                                          unique_districts=UNIQUE_DISTRICTS,
                                          get_monthly_avg=True)

 11%|█         | 2/19 [00:55<07:50, 27.70s/it]

Missing Date Found:  {Timestamp('2020-03-08 02:00:00', freq='H')}


 74%|███████▎  | 14/19 [02:49<00:52, 10.44s/it]

Missing Date Found:  {Timestamp('2021-03-14 02:00:00', freq='H')}


100%|██████████| 19/19 [04:02<00:00, 12.77s/it]


Имеется пара районов города в которых поездки не совершались, такие районы исключим

**Мы не можем прогнозировать для районов 103 и 104 (нет данных)**, а для дальнейшего анализа данные ряды будут являться лишь шумом

In [7]:
# Найдем районы имеющие только нулевые значения 
print('Districts with only 0 values:')
for district in trips_df['PULocationID'].unique():
    if trips_df[trips_df['PULocationID'] == district]['n_trips'].sum() == 0:
        print(district)

Districts with only 0 values:
103
104


In [8]:
# Исключим районы 103 и 104
trips_df = trips_df[~trips_df['PULocationID'].isin([103, 104])]
mean_trips_df = mean_trips_df[~mean_trips_df['PULocationID'].isin([103, 104])]

### Сохранение Результатов 

In [ ]:
# Сохраняем обработанные данные
os.chdir(PROJECT_SOURCE+'/processed_data')

trips_df.to_csv('processed_ts_main.csv', index=False)
mean_trips_df.to_csv('processed_ts_monthly.csv', index=False)